# Feature Engineering

**Objetivo**: Partir del análisis descriptivo hecho sobre las variables macro-económicas del Argentina, para crear nuevas features que permitan predecir el MERVAL.

## Tabla de contenidos
1. [Config](#config)
2. [Input — Carga de datos](#input)
3. [Utils](#utils)
4. [FE](#fe)
   - [A. Filtros (media móvil y pasa-altos)](#fe-a)
   - [B. Transformaciones FFT en ventanas previas](#fe-b)
   - [C. Deltas de frecuencias](#fe-c)
   - [D. Lags y otros deltas](#fe-d)
5. [Output — Export raw dataset](#output)
   - [Quick checks](#quick-checks)
   - [Saving FE dataset](#saving-fe)

### Config <a id="config"></a>
Parámetros globales, rutas, *seed*, tamaño de ventanas y opciones para evitar *data leakage*.


In [ ]:
import os
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

DATASET_NAME = "dataset_v2"

# Ruta del dataset original:
DATA_PATH = f"""../inputs/{DATASET_NAME}.csv"""

# Salida de features generadas
OUTPUT_FEATURES_PATH = f"""../features/{DATASET_NAME}_fe.csv"""

# Salida de dataset con features generadas
OUTPUT_PATH = f"""../inputs/{DATASET_NAME}_with_fe.csv"""

# Columna de fecha
DATE_COL = "fecha"

# Rando de fechas a utilizar:
DATE_START = "2022-12-01" # Para que el corte inicial sea desde ene-2023
DATE_END = "2025-08-31" # Estaba tomando hasta sep-2025 pero el emae llega hasta ago-2025

# Variable objetivo y horizonte de forecast (ejemplo: retorno del MERVAL t+1)
TARGET_FEATURES = ["merval_apertura", "merval_maximo", "merval_minimo", "merval_cierre"]
H = 1   # horizonte en pasos

# Lista de features pre-seleccionadas
SELECTED_FEATURES = [# REVISAR CON IAN & EUGENIO
    "emae_original", "emae_desestacionalizado", "emae_tendencia_ciclo",
    # Balanza de pagos:
    "badlar", 
    # Tipo de cambio:
    "tc_mayorista", "tc_minorista",
    # Agregados / circulación monetaria:
    "base_monetaria", "m1", "m2", "m2_transaccional",
    # Préstamos (en pesos) y tasas:
    "prestamos_privados_ars", "tasa_prestamos_personales",
    # Inflación o IPC:
    "inflacion",
] + TARGET_FEATURES # Aseguramos incluir la target

# Columnas a excluir (IDs, texto libre, etc.)
EXCLUDE_COLS = []

pd.options.display.max_columns = None
pd.options.display.width = None

###  Input — Carga de datos <a id="input"></a>
Lectura del dataset base, normalización de índice temporal y *sanity checks* iniciales.


In [2]:
# Intentamos leer parquet/csv en orden. Ajustá DATA_PATH según tu proyecto.
df = None
if os.path.exists(DATA_PATH):
    if DATA_PATH.lower().endswith(".parquet"):
        df = pd.read_parquet(DATA_PATH)
    elif DATA_PATH.lower().endswith(".csv"):
        df = pd.read_csv(DATA_PATH)
    else:
        try:
            df = pd.read_parquet(DATA_PATH)
        except Exception:
            df = pd.read_csv(DATA_PATH)
else:
    print(f"[ADVERTENCIA] No se encontró DATA_PATH: {DATA_PATH}")

# Si hay columna de fecha, la usamos como índice temporal
if df is not None and DATE_COL and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.sort_values(DATE_COL).set_index(DATE_COL)

# Filtramos por rango de fechas si se especifica
if df is not None and DATE_START and DATE_END:
    df = df.loc[(df.index >= DATE_START) & (df.index <= DATE_END)]

# Ordenamos por fecha (índice)
if df is not None:
    df = df.sort_index()

print("Forma del df:", None if df is None else df.shape)
print("Columnas:", None if df is None else list(df.columns)[:10], "...")

Forma del df: (679, 36)
Columnas: ['merval_apertura', 'merval_maximo', 'merval_minimo', 'merval_cierre', 'embi_spread_arg', 'embi_spreads_brz', 'embi_spread_global', 'emae_original', 'emae_desestacionalizado', 'emae_tendencia_ciclo'] ...


In [3]:
print(f"Min date: {df.index.min()}")
display(df.head(3))
print(f"Max date: {df.index.max()}")
display(df.tail(3))

Min date: 2022-12-01 00:00:00


,merval_apertura,merval_maximo,merval_minimo,merval_cierre,embi_spread_arg,embi_spreads_brz,embi_spread_global,emae_original,emae_desestacionalizado,emae_tendencia_ciclo,tc_minorista,tc_mayorista,rrii,tamar,badlar,tm20,tasa_prestamos_personales,tasa_pf_pesos,tasa_pf_dolares,base_monetaria,circulacion_monetaria,dep_ccorrientes,dep_cahorro,dep_plazo,dep_plazo_fijo,m1,m2,m2_transaccional,dep_dolar_priv_ars,prestamos_privados_pesos_ars,prestamos_privados_dolares_ars,prestamos_privados_ars,inflacion,rem,cer,uva
fecha,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2022-12-01,168525.2656,171189.0,166693.0,170604.2656,2212.262,250.317,387.332,146.180808,148.762438,149.717452,174.86,167.69,38188.2,NaN,68.0625,66.0000,79.98,68.781686,0.375802,4550659.0,3455102.0,3207476.0,3913802.0,10057344.0,7469663.0,6662578.0,11159175.0,8253928.868,17387430.36,6679941.0,597797.025,7277738.025,5.1,98.4,696902.0,175.63
2022-12-02,170664.0625,171463.0,169347.0,169691.7344,2247.765,247.389,387.172,146.180808,148.762438,149.717452,175.42,168.09,38294.3,NaN,68.3125,66.1875,81.75,68.937410,0.330453,4413760.0,3485784.0,3147481.0,3985426.0,10132099.0,7568278.0,6633265.0,11166250.0,8250986.186,17510346.17,6659982.0,600093.081,7260075.081,5.1,98.4,698322.0,175.99
2022-12-05,169697.4531,171247.0,164337.0,164467.2813,2295.579,243.166,380.339,146.180808,148.762438,149.717452,176.44,169.14,38567.7,NaN,68.8125,65.1875,82.37,69.135988,0.329961,4618287.0,3540447.0,3235673.0,3896968.0,9984042.0,7444902.0,6776120.0,11163327.0,8328357.743,17414186.27,6623979.0,596539.145,7220518.145,5.1,98.4,702602.0,176.35


Max date: 2025-08-29 00:00:00


,merval_apertura,merval_maximo,merval_minimo,merval_cierre,embi_spread_arg,embi_spreads_brz,embi_spread_global,emae_original,emae_desestacionalizado,emae_tendencia_ciclo,tc_minorista,tc_mayorista,rrii,tamar,badlar,tm20,tasa_prestamos_personales,tasa_pf_pesos,tasa_pf_dolares,base_monetaria,circulacion_monetaria,dep_ccorrientes,dep_cahorro,dep_plazo,dep_plazo_fijo,m1,m2,m2_transaccional,dep_dolar_priv_ars,prestamos_privados_pesos_ars,prestamos_privados_dolares_ars,prestamos_privados_ars,inflacion,rem,cer,uva
fecha,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-08-27,2033345.250,2034490.0,1966580.0,1974150.625,852.821,198.881,272.314,150.831536,151.449169,151.501857,1370.79,1359.42,41255.0,61.5625,54.00,56.7500,74.77,54.321843,1.439131,44202019.0,23604996.0,17510057.0,23572830.0,76086006.0,47997905.0,41115053.0,75959382.0,51055688.56,134293596.9,79469805.0,24262869.26,103732674.3,1.9,20.9,6206450.0,1566.38
2025-08-28,1974150.625,2044300.0,1973560.0,1997378.750,836.734,196.856,270.477,150.831536,151.449169,151.501857,1349.44,1343.67,40963.0,64.8750,58.25,60.8125,77.83,58.376169,1.962373,44773011.0,23619367.0,17754026.0,24565656.0,76459743.0,48488402.0,41373393.0,77349942.0,51932219.64,135616949.4,79830078.0,24184656.93,104014734.9,1.9,20.9,6210219.0,1567.33
2025-08-29,1997378.750,2001560.0,1962870.0,1984845.000,828.532,192.569,269.377,150.831536,151.449169,151.501857,1361.42,1323.83,39986.0,63.6875,57.25,59.6875,76.74,57.870422,1.953966,44259863.0,23640833.0,17718930.0,27546070.0,76633744.0,49158653.0,41359763.0,78051378.0,54267966.62,138569966.1,79921235.0,24198348.89,104119583.9,1.9,20.9,6213991.0,1568.28


Filtramos el df con las features a utilizar (selected + target)

In [4]:
df = df[SELECTED_FEATURES]

Reset del index para trabajar a las series temporales sin preocuparnos con fines de semanas o feriados

In [5]:
df.reset_index(inplace=True, drop=False)

Limpiamos nulls que puedan quedar

In [6]:
df.isnull().sum().sum()

125

In [7]:
df.loc[df.badlar.isnull()]

,fecha,emae_original,emae_desestacionalizado,emae_tendencia_ciclo,badlar,tc_mayorista,tc_minorista,base_monetaria,m1,m2,m2_transaccional,prestamos_privados_ars,tasa_prestamos_personales,inflacion,merval_apertura,merval_maximo,merval_minimo,merval_cierre
184,2023-09-02,148.432243,149.530299,146.204658,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.7,6.349992e+05,641501.0,632175.0,6.332433e+05
214,2023-10-14,147.506961,148.245833,145.642650,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.3,7.012052e+05,734135.0,637483.0,6.541364e+05
292,2024-02-12,133.637222,143.718698,143.915926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.2,1.128511e+06,1146480.0,1094040.0,1.105580e+06
321,2024-03-24,142.568626,142.767894,143.843167,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,2.501153e+06,2554800.0,2501150.0,2.551020e+06
345,2024-05-01,156.776591,142.560149,144.342531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27671597.29,NaN,4.2,1.323586e+06,1372560.0,1323590.0,1.369674e+06
414,2024-08-10,147.327539,147.656882,146.492732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.2,1.738556e+06,1738560.0,1681770.0,1.683316e+06
415,2024-08-11,147.327539,147.656882,146.492732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.2,1.683316e+06,1789040.0,1715180.0,1.764128e+06
426,2024-08-24,147.327539,147.656882,146.492732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.2,1.606967e+06,1627370.0,1580600.0,1.593002e+06
478,2024-11-06,148.069531,150.616130,149.218443,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.4,1.931281e+06,2016580.0,1931280.0,1.976613e+06
511,2024-12-24,148.566816,152.288769,149.992622,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.7,2.501153e+06,2554800.0,2501150.0,2.551020e+06


Los nulls que persisten corresponde a fines de semana o feriados

Los elimino

In [8]:
df.dropna(inplace=True, axis=0)

In [9]:
df.isnull().sum().sum()

0

### Utils <a id="utils"></a>
Funciones auxiliares (rolling seguros, alineación, manejo de bordes, helpers FFT, validaciones).

In [10]:
import math

def ensure_series(x):
    return x.dropna().astype(float)

# A. Filtros (media móvil y pasa-altos)
def moving_average(s, window: int = 5, min_periods: int = 1):
    s = ensure_series(s)
    return s.rolling(window=window, min_periods=min_periods).mean()

def high_pass_via_ema(s, span: int = 20):
    """
    Alta-paso simple: quita la componente de baja frecuencia restando la EMA.
    """
    s = ensure_series(s)
    ema = s.ewm(span=span, adjust=False).mean()
    return s - ema

def high_pass_via_rolling_mean(s, window: int = 20):
    s = ensure_series(s)
    ma = s.rolling(window=window, min_periods=1).mean()
    return s - ma

def pct_change_safe(s, periods: int = 1):
    s = ensure_series(s)
    return s.pct_change(periods=periods).replace([np.inf, -np.inf], np.nan)

def diff_safe(s, periods: int = 1):
    s = ensure_series(s)
    return s.diff(periods=periods)

def rolling_std(s, window: int = 20, min_periods: int = 5):
    s = ensure_series(s)
    return s.rolling(window=window, min_periods=min_periods).std()

# B. Transformaciones FFT en ventanas previas
EPS = 1e-12

def spectral_feats(x, fs=1.0):
    x = np.asarray(x, dtype=float)
    # ventana sin información útil
    if np.isnan(x).all() or np.allclose(np.nanstd(x), 0.0):
        return {"dom_freq": np.nan, "spec_centroid": np.nan, "spec_entropy": 0.0}

    # estabiliza: quitar media (detrend simple)
    x = x - np.nanmean(x)

    X = np.fft.rfft(x)
    Pxx = (np.abs(X) ** 2).astype(float)
    psum = Pxx.sum()

    if psum <= EPS:
        return {"dom_freq": np.nan, "spec_centroid": np.nan, "spec_entropy": 0.0}

    f = np.fft.rfftfreq(len(x), d=1.0/fs)

    k = int(np.nanargmax(Pxx))
    dom_freq = float(f[k])

    spec_centroid = float((f * Pxx).sum() / (psum + EPS))

    p = Pxx / (psum + EPS)
    p_safe = np.clip(p, EPS, 1.0)
    spec_entropy = float(-(p_safe * np.log(p_safe)).sum() / np.log(len(p_safe)))

    return {"dom_freq": dom_freq, "spec_centroid": spec_centroid, "spec_entropy": spec_entropy}

def _pad_left(x, target_len, mode="reflect"):
    x = np.asarray(x, float)
    if len(x) >= target_len:
        return x[-target_len:]
    need = target_len - len(x)
    if mode == "reflect":
        pad = np.flip(x[1:min(len(x), need+1)])
        if len(pad) < need:
            pad = np.pad(pad, (0, need-len(pad)), mode="edge")
    elif mode == "edge":
        pad = np.full(need, x[0])
    else:
        pad = np.zeros(need)
    return np.concatenate([pad, x])

def _ffill_within_array(x, max_steps=3):
    x = x.copy()
    mask = np.isnan(x)
    if mask.any():
        for i in range(1, min(max_steps, len(x)) + 1):
            to_fill = mask & (~np.isnan(np.roll(x, i)))
            x[to_fill] = np.roll(x, i)[to_fill]
            mask = np.isnan(x)
            if not mask.any():
                break
        if mask.any():
            fv = np.flatnonzero(~np.isnan(x))
            if fv.size:
                x[:fv[0]] = x[fv[0]]
    return x

def rolling_fft_features(
    s: pd.Series,
    window: int = 24,
    min_periods: int = None,   # por defecto = window
    align: str = 'right',      # 'right' = causal, no mira futuro
    pad_left: str = None       # None | 'repeat' | 'reflect'
):
    """
    Devuelve DataFrame con:
      - spec_centroid: sum(f*P)/sum(P)
      - dom_freq: argmax(P) (excluyendo DC)
    Frecuencia en ciclos/mes (fs=1).
    """
    s = s.astype(float).copy()
    s = s.sort_index()

    if pad_left in ('repeat', 'reflect'):
        need = max(0, window - s.notna().idxmax().month)  # heurística simple
        # Mejor: calcular cuántos NaNs iniciales reales hay
        k = s.isna().cummin().sum()  # cantidad al principio, robusto
        k = int(k)
        if k > 0:
            first_val = s.dropna().iloc[0]
            if pad_left == 'repeat':
                fill = pd.Series([first_val]*k, index=pd.date_range(
                    s.index[0] - pd.offsets.MonthEnd(k),
                    periods=k, freq='M'
                ))
            else: # reflect
                vals = s.dropna().values[:min(window, len(s.dropna()))]
                ref = np.concatenate((vals[1:][::-1], vals[:1]))  # simple espejo
                ref = ref[:k] if k <= len(ref) else np.pad(ref, (0, k-len(ref)), mode='edge')
                fill = pd.Series(ref, index=pd.date_range(
                    s.index[0] - pd.offsets.MonthEnd(k),
                    periods=k, freq='M'
                ))
            s = pd.concat([fill, s]).sort_index()

    if align != 'right':
        raise ValueError("Para forecasting usar align='right' (causal).")

    if min_periods is None:
        min_periods = window

    idx = s.index
    spec_centroid = pd.Series(index=idx, dtype=float)
    dom_freq = pd.Series(index=idx, dtype=float)

    fs = 1.0  # 1 muestra/mes
    # ventana deslizante causal: usamos los últimos 'window' puntos para cada t
    for t in range(len(s)):
        lo = max(0, t - window + 1)
        win = s.iloc[lo:t+1].dropna()
        if len(win) < min_periods:
            continue

        x = win.values
        # FFT real, potencia y frecuencias
        X = np.fft.rfft(x - x.mean())  # quitamos DC parcial
        P = (X * np.conj(X)).real
        f = np.fft.rfftfreq(len(x), d=1.0/fs)

        # Excluir DC (f==0) para dom_freq
        if len(P) > 1:
            P_no_dc = P.copy()
            P_no_dc[0] = 0.0
        else:
            P_no_dc = P

        # centroid
        denom = P.sum()
        sc = np.nan if denom == 0 else (f * P).sum() / denom
        # dominante
        df = f[np.argmax(P_no_dc)] if len(P_no_dc) else np.nan

        spec_centroid.iloc[t] = sc
        dom_freq.iloc[t] = df

    out = pd.DataFrame({
        'spec_centroid': spec_centroid,
        'dom_freq': dom_freq
    })
    return out

# C. Deltas de frecuencias
# def diff_blocksafe(s: pd.Series, lag: int) -> pd.Series:
#     d = s.diff(lag)
#     # primer valor válido tras un bloque de NaNs: poner 0 (no hay cambio previo conocido)
#     first_after_gap = s.notna() & s.shift(lag).isna()
#     d = d.where(~first_after_gap, 0.0)
#     # si cualquiera de los lados es NaN (huecos internos), dejamos NaN (no inventamos)
#     d = d.where(~(s.isna() | s.shift(lag).isna()))
#     return d

def diff_causal(s: pd.Series, lag: int) -> pd.Series:
    d = s.diff(lag)
    return d

# D. Lags y otros deltas
def make_lagged(df, cols, lags=(1, 5, 10, 20)):
    out = {}
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        for L in lags:
            out[f"{c}_lag{L}"] = s.shift(L)
    return pd.DataFrame(out, index=df.index)

def make_deltas(df, cols, periods=(1, 5, 10)):
    out = {}
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        for p in periods:
            out[f"{c}_diff{p}"] = diff_safe(s, p)
            out[f"{c}_pct{p}"] = pct_change_safe(s, p)
    return pd.DataFrame(out, index=df.index)

### FE <a id="fe"></a>

#### A. Filtros (media móvil y pasa-altos) <a id="fe-a"></a>
Construcción de versiones suavizadas y residuales (*high-pass*) para resaltar señales de corto plazo.


In [11]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    base_cols = [c for c in SELECTED_FEATURES if c in df.columns]
    fe_ma = pd.DataFrame(index=df.index)
    fe_hp = pd.DataFrame(index=df.index)

    for c in base_cols:
        fe_ma[f"{c}_ma_3"] = moving_average(df[c], 3)
        fe_ma[f"{c}_ma_5"] = moving_average(df[c], 5)
        fe_hp[f"{c}_hp_ema20"] = high_pass_via_ema(df[c], span=20)
        fe_hp[f"{c}_hp_rm20"] = high_pass_via_rolling_mean(df[c], window=20)

    FE_FILTERS = pd.concat([fe_ma, fe_hp], axis=1)
    print("FE_FILTERS shape:", FE_FILTERS.shape)


FE_FILTERS shape: (665, 68)


In [12]:
FE_FILTERS.describe(include='all')

,emae_original_ma_3,emae_original_ma_5,emae_desestacionalizado_ma_3,emae_desestacionalizado_ma_5,emae_tendencia_ciclo_ma_3,emae_tendencia_ciclo_ma_5,badlar_ma_3,badlar_ma_5,tc_mayorista_ma_3,tc_mayorista_ma_5,tc_minorista_ma_3,tc_minorista_ma_5,base_monetaria_ma_3,base_monetaria_ma_5,m1_ma_3,m1_ma_5,m2_ma_3,m2_ma_5,m2_transaccional_ma_3,m2_transaccional_ma_5,prestamos_privados_ars_ma_3,prestamos_privados_ars_ma_5,tasa_prestamos_personales_ma_3,tasa_prestamos_personales_ma_5,inflacion_ma_3,inflacion_ma_5,merval_apertura_ma_3,merval_apertura_ma_5,merval_maximo_ma_3,merval_maximo_ma_5,merval_minimo_ma_3,merval_minimo_ma_5,merval_cierre_ma_3,merval_cierre_ma_5,emae_original_hp_ema20,emae_original_hp_rm20,emae_desestacionalizado_hp_ema20,emae_desestacionalizado_hp_rm20,emae_tendencia_ciclo_hp_ema20,emae_tendencia_ciclo_hp_rm20,badlar_hp_ema20,badlar_hp_rm20,tc_mayorista_hp_ema20,tc_mayorista_hp_rm20,tc_minorista_hp_ema20,tc_minorista_hp_rm20,base_monetaria_hp_ema20,base_monetaria_hp_rm20,m1_hp_ema20,m1_hp_rm20,m2_hp_ema20,m2_hp_rm20,m2_transaccional_hp_ema20,m2_transaccional_hp_rm20,prestamos_privados_ars_hp_ema20,prestamos_privados_ars_hp_rm20,tasa_prestamos_personales_hp_ema20,tasa_prestamos_personales_hp_rm20,inflacion_hp_ema20,inflacion_hp_rm20,merval_apertura_hp_ema20,merval_apertura_hp_rm20,merval_maximo_hp_ema20,merval_maximo_hp_rm20,merval_minimo_hp_ema20,merval_minimo_hp_rm20,merval_cierre_hp_ema20,merval_cierre_hp_rm20
count,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,665.000000,665.000000,665.000000,665.000000,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,6.650000e+02,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000,665.000000
mean,148.457876,148.450883,148.206479,148.202438,147.871346,147.868662,64.358208,64.376667,722.566170,720.790406,747.709213,745.920191,1.734217e+07,1.728225e+07,2.203627e+07,2.198430e+07,4.081725e+07,4.071881e+07,2.724802e+07,2.718311e+07,3.771248e+07,3.756733e+07,87.914409,87.922269,6.790827,6.795639,1.289300e+06,1.286532e+06,1.314380e+06,1.311574e+06,1.269104e+06,1.266388e+06,1.291832e+06,1.289088e+06,0.073877,0.066439,0.037625,0.038382,0.025742,0.025491,-0.285426,-0.248824,16.481297,16.536861,16.627344,16.676736,5.516452e+05,5.637825e+05,5.017212e+05,4.986176e+05,9.332006e+05,9.273658e+05,6.267626e+05,6.186699e+05,1.347610e+06,1.356516e+06,-0.079701,-0.086943,-0.045775,-0.045714,28055.240939,27981.636945,28428.953141,28336.370923,27538.923084,27413.976762,27808.958000,27664.807355
std,6.277957,6.230506,3.261809,3.252988,2.636233,2.632608,32.717970,32.691587,380.176089,379.835304,387.954171,387.698511,1.175723e+07,1.171349e+07,1.218740e+07,1.217776e+07,2.215558e+07,2.213669e+07,1.531859e+07,1.530126e+07,2.993922e+07,2.985088e+07,23.811479,23.729091,5.360058,5.339033,7.956835e+05,7.959589e+05,8.089901e+05,8.093082e+05,7.821213e+05,7.823889e+05,7.946531e+05,7.949511e+05,2.689867,3.205496,0.748194,0.854875,0.265545,0.274824,5.214106,5.866506,36.927616,43.397205,37.169189,43.887474,1.056484e+06,1.161106e+06,5.770326e+05,6.663120e+05,1.381846e+06,1.609818e+06,1.252864e+06,1.443491e+06,1.067481e+06,1.101354e+06,6.129843,6.954287,1.575753,1.831172,82918.178420,98036.997357,83327.946394,98454.823428,81623.450618,96625.801728,83121.626515,98316.463397
min,133.637222,133.637222,140.975690,140.975690,143.843167,143.843167,27.625000,27.675000,167.690000,167.690000,174.860000,174.860000,4.482210e+06,4.482210e+06,6.647922e+0

#### B. Transformaciones FFT en ventanas previas <a id="fe-b"></a>
Espectro en ventanas deslizantes: amplitudes, energía por bandas, picos dominantes y *ratios*.

In [13]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    base_cols = [c for c in SELECTED_FEATURES if c in df.columns]

    fft_frames = []
    for c in base_cols:
        # Nota: ajustá window/min_periods según tu política (p.ej., 24)
        out = rolling_fft_features(df[c], window=20, min_periods=20, align='right')
        # renombrar para evitar duplicados
        out = out.rename(columns={
            "spec_centroid": f"{c}_spec_centroid",
            "dom_freq": f"{c}_dom_freq"
        })
        fft_frames.append(out)

    # concatenar una sola vez (mejor performance) y alinear
    FE_FFT = pd.concat(fft_frames, axis=1) if fft_frames else pd.DataFrame(index=df.index)

    print("FE_FFT shape:", FE_FFT.shape)


FE_FFT shape: (665, 34)


In [14]:
FE_FFT.describe(include='all')

,emae_original_spec_centroid,emae_original_dom_freq,emae_desestacionalizado_spec_centroid,emae_desestacionalizado_dom_freq,emae_tendencia_ciclo_spec_centroid,emae_tendencia_ciclo_dom_freq,badlar_spec_centroid,badlar_dom_freq,tc_mayorista_spec_centroid,tc_mayorista_dom_freq,tc_minorista_spec_centroid,tc_minorista_dom_freq,base_monetaria_spec_centroid,base_monetaria_dom_freq,m1_spec_centroid,m1_dom_freq,m2_spec_centroid,m2_dom_freq,m2_transaccional_spec_centroid,m2_transaccional_dom_freq,prestamos_privados_ars_spec_centroid,prestamos_privados_ars_dom_freq,tasa_prestamos_personales_spec_centroid,tasa_prestamos_personales_dom_freq,inflacion_spec_centroid,inflacion_dom_freq,merval_apertura_spec_centroid,merval_apertura_dom_freq,merval_maximo_spec_centroid,merval_maximo_dom_freq,merval_minimo_spec_centroid,merval_minimo_dom_freq,merval_cierre_spec_centroid,merval_cierre_dom_freq
count,635.000000,646.000000,634.000000,646.000000,635.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,636.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000,646.000000
mean,0.119275,0.049923,0.119692,0.050542,0.120031,0.051316,0.193992,0.151858,0.119508,0.057895,0.123504,0.063932,0.128924,0.069582,0.126880,0.064551,0.107973,0.059752,0.085172,0.051935,0.105785,0.057353,0.180620,0.114551,0.115084,0.050077,0.116148,0.063777,0.110790,0.061300,0.110836,0.061455,0.117166,0.063390
std,0.060472,0.027066,0.060679,0.030685,0.062152,0.037279,0.074377,0.144527,0.039236,0.048381,0.042346,0.068705,0.050022,0.062997,0.035958,0.035399,0.034875,0.027381,0.024098,0.012141,0.029035,0.017939,0.054666,0.096635,0.062833,0.035383,0.036982,0.036777,0.034418,0.029268,0.033923,0.028536,0.037149,0.035692
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.075356,0.050000,0.066246,0.050000,0.074071,0.050000,0.056304,0.050000,0.068089,0.050000,0.060652,0.050000,0.056205,0.050000,0.056321,0.050000,0.068377,0.050000,0.000000,0.000000,0.059119,0.050000,0.054835,0.050000,0.058126,0.050000,0.058386,0.050000
25%,0.088462,0.050000,0.088462,0.050000,0.088462,0.050000,0.124719,0.050000,0.104076,0.050000,0.103162,0.050000,0.091881,0.050000,0.099472,0.050000,0.083530,0.050000,0.070618,0.050000,0.083572,0.050000,0.136260,0.050000,0.088462,0.050000,0.088068,0.050000,0.083372,0.050000,0.084711,0.050000,0.087875,0.050000
50%,0.094199,0.050000,0.094199,0.050000,0.094199,0.050000,0.191909,0.050000,0.109003,0.050000,0.113069,0.050000,0.116563,0.050000,0.119176,0.050000,0.096742,0.050000,0.077410,0.050000,0.095993,0.050000,0.184380,0.050000,0.094199,0.050000,0.108492,0.050000,0.105192,0.050000,0.105503,0.050000,0.110481,0.050000
75%,0.140910,0.050000,0.140910,0.050000,0.140910,0.050000,0.257285,0.250000,0.114573,0.050000,0.127216,0.050000,0.150945,0.050000,0.147441,0.050000,0.124464,0.050000,0.090763,0.050000,0.125115,0.050000,0.220729,0.200000,0.140910,0.050000,0.138804,0.050000,0.131187,0.050000,0.129232,0.050000,0.139520,0.050000
max,0.365107,0.500000,0.347344,0.500000,0.365474,0.500000,0.389722,0.500000,0.338889,0.500000,0.350596,0.500000,0.293260,0.500000,0.271949,0.400000,0.255122,0.250000,0.242321,0.250000,0.205017,0.150000,0.325922,0.500000,0.352291,0.500000,0.267722,0.400000,0.251144,0.300000,0.241769,0.250000,0.267722,0.400000


### C. Deltas de frecuencias <a id="fe-c"></a>
Diferencias/variaciones entre bandas o picos de frecuencia en ventanas sucesivas.

In [15]:
if df is None or 'FE_FFT' not in globals() or not isinstance(FE_FFT, pd.DataFrame) or FE_FFT.empty:
    print("[INFO] Saltando: no hay FE_FFT.")
    FE_FREQ_DELTAS = pd.DataFrame(index=(df.index if df is not None else None))
else:
    idx = FE_FFT.index
    out = {}

    # Deltas entre bandas (p. ej., potencia banda0 - banda1, banda1 - banda2)
    for col in FE_FFT.columns:
        if col.endswith("_band_power_0"):
            base = col.rsplit("_band_power_", 1)[0]
            b0 = pd.to_numeric(FE_FFT.get(f"{base}_band_power_0"), errors="coerce")
            b1 = pd.to_numeric(FE_FFT.get(f"{base}_band_power_1"), errors="coerce")
            b2 = pd.to_numeric(FE_FFT.get(f"{base}_band_power_2"), errors="coerce")

            if b0 is not None and b1 is not None:
                out[f"{base}_bp01"] = b0 - b1
            if b1 is not None and b2 is not None:
                out[f"{base}_bp12"] = b1 - b2

    # Deltas temporales sobre métricas espectrales por variable (diff de 1 y 3 pasos)
    for suf in ("_dom_freq", "_spec_centroid", "_spec_entropy"):
        metrics = [c for c in FE_FFT.columns if c.endswith(suf)]
        for c in metrics:
            out[f"{c}_diff1"] = FE_FFT[c].diff(1)
            out[f"{c}_diff3"] = FE_FFT[c].diff(3)

    FE_FREQ_DELTAS = pd.DataFrame(out, index=idx)
    print("FE_FREQ_DELTAS shape:", FE_FREQ_DELTAS.shape)

FE_FREQ_DELTAS shape: (665, 68)


In [16]:
FE_FREQ_DELTAS.describe(include='all')

,emae_original_dom_freq_diff1,emae_original_dom_freq_diff3,emae_desestacionalizado_dom_freq_diff1,emae_desestacionalizado_dom_freq_diff3,emae_tendencia_ciclo_dom_freq_diff1,emae_tendencia_ciclo_dom_freq_diff3,badlar_dom_freq_diff1,badlar_dom_freq_diff3,tc_mayorista_dom_freq_diff1,tc_mayorista_dom_freq_diff3,tc_minorista_dom_freq_diff1,tc_minorista_dom_freq_diff3,base_monetaria_dom_freq_diff1,base_monetaria_dom_freq_diff3,m1_dom_freq_diff1,m1_dom_freq_diff3,m2_dom_freq_diff1,m2_dom_freq_diff3,m2_transaccional_dom_freq_diff1,m2_transaccional_dom_freq_diff3,prestamos_privados_ars_dom_freq_diff1,prestamos_privados_ars_dom_freq_diff3,tasa_prestamos_personales_dom_freq_diff1,tasa_prestamos_personales_dom_freq_diff3,inflacion_dom_freq_diff1,inflacion_dom_freq_diff3,merval_apertura_dom_freq_diff1,merval_apertura_dom_freq_diff3,merval_maximo_dom_freq_diff1,merval_maximo_dom_freq_diff3,merval_minimo_dom_freq_diff1,merval_minimo_dom_freq_diff3,merval_cierre_dom_freq_diff1,merval_cierre_dom_freq_diff3,emae_original_spec_centroid_diff1,emae_original_spec_centroid_diff3,emae_desestacionalizado_spec_centroid_diff1,emae_desestacionalizado_spec_centroid_diff3,emae_tendencia_ciclo_spec_centroid_diff1,emae_tendencia_ciclo_spec_centroid_diff3,badlar_spec_centroid_diff1,badlar_spec_centroid_diff3,tc_mayorista_spec_centroid_diff1,tc_mayorista_spec_centroid_diff3,tc_minorista_spec_centroid_diff1,tc_minorista_spec_centroid_diff3,base_monetaria_spec_centroid_diff1,base_monetaria_spec_centroid_diff3,m1_spec_centroid_diff1,m1_spec_centroid_diff3,m2_spec_centroid_diff1,m2_spec_centroid_diff3,m2_transaccional_spec_centroid_diff1,m2_transaccional_spec_centroid_diff3,prestamos_privados_ars_spec_centroid_diff1,prestamos_privados_ars_spec_centroid_diff3,tasa_prestamos_personales_spec_centroid_diff1,tasa_prestamos_personales_spec_centroid_diff3,inflacion_spec_centroid_diff1,inflacion_spec_centroid_diff3,merval_apertura_spec_centroid_diff1,merval_apertura_spec_centroid_diff3,merval_maximo_spec_centroid_diff1,merval_maximo_spec_centroid_diff3,merval_minimo_spec_centroid_diff1,merval_minimo_spec_centroid_diff3,merval_cierre_spec_centroid_diff1,merval_cierre_spec_centroid_diff3
count,6.450000e+02,6.430000e+02,6.450000e+02,643.000000,6.450000e+02,643.000000,645.000000,643.000000,6.450000e+02,6.430000e+02,6.450000e+02,6.430000e+02,6.450000e+02,6.430000e+02,645.000000,643.000000,645.000000,6.430000e+02,6.450000e+02,6.430000e+02,6.450000e+02,6.430000e+02,6.450000e+02,6.430000e+02,6.450000e+02,643.000000,6.450000e+02,6.430000e+02,6.450000e+02,6.430000e+02,6.450000e+02,6.430000e+02,645.000000,6.430000e+02,628.000000,622.000000,629.000000,620.000000,6.290000e+02,6.230000e+02,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,6.300000e+02,623.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000,645.000000,643.000000
mean,-8.606380e-20,1.294972e-19,4.303190e-20,-0.000311,1.290957e-19,-0.000078,-0.000698,-0.002100,1.936436e-19,8.633149e-20,-6.454785e-20,-2.805774e-19,4.303190e-20,-1.079144e-19,0.000078,0.000233,0.000000,1.726630e-19,-2.151595e-20,4.316575e-20,-2.151595e-20,4.316575e-20,1.290957e-19,-3.453260e-19,-1.721276e-19,-0.000467,8.606380e-20,8.633149e-20,8.606380e-20,-2.158287e-20,-4.303190e-20,-8.633149e-20,0.000000,-2.158287e-20,-0.000438,-0.000227,0.000437,0.000227,-1.103163e-18,4.455149e-19,-0.000320,-0.000967,-0.000033,-0.000072,-0.000062,-0.000107,0.000101,0.000272,0.000065,0.000290,-0.000021,-0.000007,-0.000051,-0.000109,-0.000012,0.000031,0.000096,0.000128,-2.863671e-19,-0.000705,-0.000002,0.000035,0.000002,0.000014,0.000008,0.000044,0.000005,0.000047
std,3.890983e-02,3.936806e-02,4.298635e-02,0.044124,5.145401e-02,0.053280,0.098136,0.131488,4.216575e-02,6.024353e-02,3.176975e-02,4.889750e-02,4.352482e-02,5.887047e-02,0.031831,0.042000,0.024131,3.406517e-02,1.246112e-02,1.603151e-02,1.551397e-02

### D. Lags y otros deltas <a id="fe-d"></a>
Lags, *rate of change*, *z-scores* rolling, y otras transformaciones temporales.

In [17]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    base_cols = [c for c in SELECTED_FEATURES if c in df.columns]
    FE_LAGS = make_lagged(df, base_cols, lags=(1,2,3))
    FE_DELTAS = make_deltas(df, base_cols, periods=(1,3,10))
    FE_ROLLSTD = pd.DataFrame({
        f"{c}_std20": rolling_std(df[c], window=20)
        for c in base_cols
    })
    print("FE_LAGS:", FE_LAGS.shape, "FE_DELTAS:", FE_DELTAS.shape, "FE_ROLLSTD:", FE_ROLLSTD.shape)


FE_LAGS: (665, 51) FE_DELTAS: (665, 102) FE_ROLLSTD: (665, 17)


In [18]:
FE_LAGS.describe(include='all')

,emae_original_lag1,emae_original_lag2,emae_original_lag3,emae_desestacionalizado_lag1,emae_desestacionalizado_lag2,emae_desestacionalizado_lag3,emae_tendencia_ciclo_lag1,emae_tendencia_ciclo_lag2,emae_tendencia_ciclo_lag3,badlar_lag1,badlar_lag2,badlar_lag3,tc_mayorista_lag1,tc_mayorista_lag2,tc_mayorista_lag3,tc_minorista_lag1,tc_minorista_lag2,tc_minorista_lag3,base_monetaria_lag1,base_monetaria_lag2,base_monetaria_lag3,m1_lag1,m1_lag2,m1_lag3,m2_lag1,m2_lag2,m2_lag3,m2_transaccional_lag1,m2_transaccional_lag2,m2_transaccional_lag3,prestamos_privados_ars_lag1,prestamos_privados_ars_lag2,prestamos_privados_ars_lag3,tasa_prestamos_personales_lag1,tasa_prestamos_personales_lag2,tasa_prestamos_personales_lag3,inflacion_lag1,inflacion_lag2,inflacion_lag3,merval_apertura_lag1,merval_apertura_lag2,merval_apertura_lag3,merval_maximo_lag1,merval_maximo_lag2,merval_maximo_lag3,merval_minimo_lag1,merval_minimo_lag2,merval_minimo_lag3,merval_cierre_lag1,merval_cierre_lag2,merval_cierre_lag3
count,664.000000,663.000000,662.000000,664.000000,663.000000,662.000000,664.000000,663.000000,662.000000,664.000000,663.000000,662.000000,664.000000,663.000000,662.000000,664.000000,663.000000,662.000000,6.640000e+02,6.630000e+02,6.620000e+02,6.640000e+02,6.630000e+02,6.620000e+02,6.640000e+02,6.630000e+02,6.620000e+02,6.640000e+02,6.630000e+02,6.620000e+02,6.640000e+02,6.630000e+02,6.620000e+02,664.000000,663.000000,662.000000,664.000000,663.000000,662.000000,6.640000e+02,6.630000e+02,6.620000e+02,6.640000e+02,6.630000e+02,6.620000e+02,6.640000e+02,6.630000e+02,6.620000e+02,6.640000e+02,6.630000e+02,6.620000e+02
mean,148.461305,148.457730,148.454145,148.205641,148.200749,148.195842,147.868566,147.863085,147.857589,64.353069,64.362274,64.377927,723.411687,722.476154,721.514003,748.565783,747.659487,746.718202,1.736173e+07,1.732038e+07,1.727977e+07,2.205944e+07,2.203031e+07,2.200148e+07,4.086156e+07,4.080652e+07,4.075342e+07,2.727545e+07,2.723826e+07,2.720229e+07,3.775827e+07,3.765834e+07,3.755853e+07,87.926461,87.941689,87.961586,6.793373,6.800754,6.808157,1.290976e+06,1.289946e+06,1.288823e+06,1.316123e+06,1.315024e+06,1.313938e+06,1.270769e+06,1.269709e+06,1.268657e+06,1.293527e+06,1.292465e+06,1.291436e+06
std,6.340858,6.344976,6.349101,3.272256,3.272296,3.272328,2.638282,2.636489,2.634679,32.776319,32.800206,32.822529,380.078643,379.599746,379.076870,387.733074,387.321335,386.854565,1.176721e+07,1.172773e+07,1.168986e+07,1.218370e+07,1.216974e+07,1.215626e+07,2.214726e+07,2.211849e+07,2.209288e+07,1.531589e+07,1.529742e+07,1.528089e+07,2.993925e+07,2.985082e+07,2.976247e+07,23.973328,23.988214,24.000876,5.386197,5.386902,5.387599,7.955890e+05,7.957461e+05,7.958219e+05,8.088604e+05,8.089754e+05,8.091026e+05,7.820276e+05,7.821405e+05,7.822617e+05,7.945823e+05,7.947108e+05,7.948690e+05
min,133.637222,133.637222,133.637222,140.975690,140.975690,140.975690,143.843167,143.843167,143.843167,27.062500,27.062500,27.062500,167.690000,167.690000,167.690000,174.860000,174.860000,174.860000,4.413760e+06,4.413760e+06,4.413760e+06,6.633265e+06,6.633265e+06,6.633265e+06,1.115918e+07,1.115918e+07,1.115918e+07,8.250986e+06,8.250986e+06,8.250986e+06,7.139749e+06,7.139749e+06,7.139749e+06,59.170000,59.170000,59.170000,1.500000,1.500000,1.500000,1.642167e+05,1.642167e+05,1.642167e+05,1.653290e+05,1.653290e+05,1.653290e+05,1.627360e+05,1.627360e+05,1.627360e+05,1.642167e+05,1.642167e+05,1.642167e+05
25%,146.180808,146.180808,146.180808,146.262475,146.262475,146.262475,145.637471,145.637471,145.637471,33.593750,33.562500,33.531250,283.930000,283.670000,283.410000,296.202500,295.885000,295.567500,6.380352e+06,6.379794e+06,6.379237e+06,1.000992e+07,9.981582e+06,9.953248e+06,1.827766e+07,1.827731e+07,1.827696e+07,1.240297e+07,1.240026e+07,1.239755e+07,1.176773e+07,1.176771e+07,1.176768e+07,69.677500,69.675000,69.672500,2.700000,2.700000,2.700000,4.704197e+05,4.704067e+05,4.703936e+05,4.749850e+05,4.747820e+05,4.745790e+05,4.635258e+05,4.635075e+05,4.634892e+05,4.694511e+

### Output — Export raw dataset <a id="output"></a>

In [19]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
    FE_ALL = None
else:
    parts = []
    for name in ["FE_FILTERS","FE_FFT","FE_FREQ_DELTAS","FE_FRACTAL","FE_LAGS","FE_DELTAS","FE_ROLLSTD"]:
        if name in globals():
            part = globals()[name]
            if isinstance(part, pd.DataFrame) and not part.empty:
                parts.append(part)
    FE_ALL = pd.concat(parts, axis=1).sort_index() if parts else pd.DataFrame(index=df.index)
    FE_ALL = FE_ALL.apply(pd.to_numeric, errors="coerce")
    FE_ALL = FE_ALL.reindex(df.index)
    # posible bug de alineación o likeage
    FE_ALL[DATE_COL] = df[DATE_COL] # agregar columna de fecha
    FE_ALL = FE_ALL.dropna(how="all") # eliminar filas sin features
    print("FE_ALL shape:", FE_ALL.shape)

FE_ALL shape: (665, 341)


#### Quick checks <a id="quick-checks"></a>
Conservación de fechas/índice, conteo de NaNs por feature, rangos con NaNs, *shape* y *dtypes*.


In [20]:
def _to_datetime_index(idx):
    if not isinstance(idx, pd.DatetimeIndex):
        try:
            idx = pd.to_datetime(idx)
        except Exception:
            pass
    return pd.DatetimeIndex(idx).sort_values().unique()

def _infer_step(dti: pd.DatetimeIndex):
    """Intenta inferir la frecuencia. Si falla, usa la mediana del delta."""
    try:
        f = pd.infer_freq(dti)
        if f is not None:
            return pd.tseries.frequencies.to_offset(f)
    except Exception:
        pass
    if len(dti) >= 2:
        deltas = np.diff(dti.values).astype('timedelta64[ns]').astype('int64')
        med = np.median(deltas)
        return pd.to_timedelta(int(med), unit='ns')
    return None

def _ranges_from_dates(dates: pd.DatetimeIndex, step=None, max_show=10):
    """Agrupa fechas consecutivas en rangos [start, end]."""
    if dates.empty:
        return []
    dates = dates.sort_values()
    if step is None:
        step = _infer_step(dates)
    ranges = []
    start = dates[0]
    prev = dates[0]
    for d in dates[1:]:
        if step is not None and (d - prev) == step:
            prev = d
            continue
        # corte de rango
        ranges.append((start, prev))
        start = d
        prev = d
    ranges.append((start, prev))
    # limitar salida
    if len(ranges) > max_show:
        head = ranges[:max_show//2]
        tail = ranges[-max_show//2:]
        return head + [("...", "...")] + tail
    return ranges

def _ranges_from_mask(index: pd.DatetimeIndex, mask: pd.Series, step=None):
    """
    Devuelve rangos [(start, end, count)] donde mask==True en 'index'.
    La continuidad se define por el 'step' esperado; si step es None, toma adyacencia posicional.
    """
    idx = pd.DatetimeIndex(index)
    if step is None:
        step = _infer_step(idx)
    ranges = []
    in_run = False
    run_start = None
    prev_t = None
    run_count = 0

    for t, is_nan in zip(idx, mask.values):
        if is_nan:
            if not in_run:
                in_run = True
                run_start = t
                prev_t = t
                run_count = 1
            else:
                # Continuidad: según step (si no hay, por adyacencia posicional)
                if step is None or (t - prev_t) == step:
                    prev_t = t
                    run_count += 1
                else:
                    ranges.append((run_start, prev_t, run_count))
                    run_start = t
                    prev_t = t
                    run_count = 1
        else:
            if in_run:
                ranges.append((run_start, prev_t, run_count))
                in_run = False
                run_start = None
                prev_t = None
                run_count = 0
    if in_run:
        ranges.append((run_start, prev_t, run_count))
    return ranges

def _ensure_dt_series(s: pd.Series) -> pd.Series:
    """Convierte a datetime con errors='coerce' y devuelve serie sin NaT."""
    s2 = pd.to_datetime(s, errors='coerce')
    return s2.dropna()

def _dedup_sort_on_date(d: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """Ordena por fecha, elimina NaT y duplica por fecha (deja la última)."""
    ds = _ensure_dt_series(d[date_col])
    d2 = d.loc[ds.index].copy()
    d2[date_col] = ds
    d2 = d2.dropna(subset=[date_col]).sort_values(date_col)
    # si hubiera duplicados de fecha, conservar el último (podés cambiar a 'first')
    d2 = d2.drop_duplicates(subset=[date_col], keep='last')
    return d2


Índices, pérdidas y top features por NaNs (usando DATE_COL)

In [21]:
# Asserts básicos
assert 'df' in globals() and isinstance(df, pd.DataFrame), "df no está disponible."
assert 'FE_ALL' in globals() and isinstance(FE_ALL, pd.DataFrame), "FE_ALL no está disponible."
assert DATE_COL in df.columns, f"{DATE_COL} no existe en df.columns"
assert DATE_COL in FE_ALL.columns, f"{DATE_COL} no existe en FE_ALL.columns"

# Versiones por fecha (sin alterar df/FE_ALL originales) 
df_by_date   = _dedup_sort_on_date(df, DATE_COL).set_index(DATE_COL)
fe_by_date   = _dedup_sort_on_date(FE_ALL, DATE_COL).set_index(DATE_COL)

# Índices de fechas 
df_idx = pd.DatetimeIndex(df_by_date.index)
fe_idx = pd.DatetimeIndex(fe_by_date.index)

base_len = len(df_idx)
fe_len   = len(fe_idx)
inter_idx = df_idx.intersection(fe_idx)
union_idx = df_idx.union(fe_idx)

# Fechas presentes en df pero ausentes en FE_ALL (potencial pérdida al hacer join inner)
missing_in_fe = df_idx.difference(fe_idx)
missing_in_df = fe_idx.difference(df_idx)

lost_ratio_in_fe = (len(missing_in_fe) / base_len) if base_len else np.nan
lost_ratio_in_df = (len(missing_in_df) / fe_len)   if fe_len   else np.nan

# Rango de fechas y gaps como rangos
step_df = _infer_step(df_idx) if ' _infer_step' in globals() else None
# Uso de _ranges_from_dates si existe; sino, muestro muestras simples
if '_ranges_from_dates' in globals():
    ranges_missing_in_fe = _ranges_from_dates(pd.DatetimeIndex(missing_in_fe), step=step_df, max_show=12)
    ranges_missing_in_df = _ranges_from_dates(pd.DatetimeIndex(missing_in_df), step=step_df, max_show=12)
else:
    ranges_missing_in_fe = [(missing_in_fe.min(), missing_in_fe.max())] if len(missing_in_fe) else []
    ranges_missing_in_df = [(missing_in_df.min(), missing_in_df.max())] if len(missing_in_df) else []

# Merge para QA por fecha
# Importante: usamos LEFT join contra df_by_date para analizar faltantes en FE_ALL
merged = df_by_date.join(fe_by_date, how="left", rsuffix="_fe")

# Columnas de FE_ALL que quedaron en merged (sin los duplicados implícitos)
fe_cols = [c for c in FE_ALL.columns if c != DATE_COL and c in merged.columns]

# Filas (fechas) con algún NaN en columnas de features (solo FE)
na_any_mask = merged[fe_cols].isna().any(axis=1) if fe_cols else pd.Series(False, index=merged.index)
dates_with_any_nan = pd.DatetimeIndex(merged.index[na_any_mask])

# Conteo de NaN por feature (sobre las fechas de df)
feature_nan_counts = merged.loc[df_idx, fe_cols].isna().sum().sort_values(ascending=False) if fe_cols else pd.Series(dtype=int)

# Top features que más “aportan” NaN
top_feats = feature_nan_counts.head(25)

# QA resumen
summary_rows = [
    ("df_rows(unique_dates)", base_len),
    ("FE_ALL_rows(unique_dates)", fe_len),
    ("intersection_rows", len(inter_idx)),
    ("union_rows", len(union_idx)),
    ("missing_in_FE_from_df", len(missing_in_fe)),
    ("missing_in_DF_from_FE", len(missing_in_df)),
    ("lost_ratio_in_FE_vs_df", round(float(lost_ratio_in_fe), 4) if pd.notna(lost_ratio_in_fe) else np.nan),
    ("lost_ratio_in_DF_vs_FE", round(float(lost_ratio_in_df), 4) if pd.notna(lost_ratio_in_df) else np.nan),
]
summary_df = pd.DataFrame(summary_rows, columns=["metric", "value"])

# Mostrar resultados clave
print("QUICK CHECKS: Índices (por fecha) y pérdidas")
display(summary_df)

print("\nRangos de fechas en df:", df_idx.min(), "→", df_idx.max())
print("Rangos de fechas en FE_ALL:", fe_idx.min() if len(fe_idx) else None, "→", fe_idx.max() if len(fe_idx) else None)

print("\nFechas de df ausentes en FE_ALL (muestras de rangos):")
for s, e in ranges_missing_in_fe:
    print(" -", s, "→", e)

print("\nFechas de FE_ALL ausentes en df (muestras de rangos):")
for s, e in ranges_missing_in_df:
    print(" -", s, "→", e)

print("\nTop features por cantidad de NaNs (en las fechas de df):")
display(top_feats.to_frame("nan_count"))


QUICK CHECKS: Índices (por fecha) y pérdidas


,metric,value
0,df_rows(unique_dates),665.0
1,FE_ALL_rows(unique_dates),665.0
2,intersection_rows,665.0
3,union_rows,665.0
4,missing_in_FE_from_df,0.0
5,missing_in_DF_from_FE,0.0
6,lost_ratio_in_FE_vs_df,0.0
7,lost_ratio_in_DF_vs_FE,0.0



Rangos de fechas en df: 2022-12-01 00:00:00 → 2025-08-29 00:00:00
Rangos de fechas en FE_ALL: 2022-12-01 00:00:00 → 2025-08-29 00:00:00

Fechas de df ausentes en FE_ALL (muestras de rangos):

Fechas de FE_ALL ausentes en df (muestras de rangos):

Top features por cantidad de NaNs (en las fechas de df):


,nan_count
emae_desestacionalizado_spec_centroid_diff3,45
emae_original_spec_centroid_diff3,43
emae_tendencia_ciclo_spec_centroid_diff3,42
inflacion_spec_centroid_diff3,42
emae_original_spec_centroid_diff1,37
emae_desestacionalizado_spec_centroid_diff1,36
emae_tendencia_ciclo_spec_centroid_diff1,36
inflacion_spec_centroid_diff1,35
emae_desestacionalizado_spec_centroid,31
emae_original_spec_centroid,30


Rangos de NaNs por feature (usando DATE_COL e índice por fecha)

In [22]:
# Índice de trabajo y step esperado
m_idx = pd.DatetimeIndex(merged.index).unique().sort_values()
merged = merged.reindex(m_idx)  # asegurar orden y unicidad temporal
step = _infer_step(m_idx) if '_infer_step' in globals() else None

# Columnas que vienen de FE_ALL en el merged (excluimos la columna de fecha por si acaso)
fe_cols = [c for c in FE_ALL.columns if c != DATE_COL and c in merged.columns]

# Cálculo de rangos por feature
feature_null_ranges = {}
feature_nan_counts = {}
feature_range_counts = {}

for c in fe_cols:
    col = merged[c]
    mask_nan = col.isna()
    n_nan = int(mask_nan.sum())
    if n_nan == 0:
        continue
    if '_ranges_from_mask' in globals():
        ranges = _ranges_from_mask(merged.index, mask_nan, step=step)
    else:
        # fallback simple si no está _ranges_from_mask
        ranges = []
        run_start = None
        for i, (ts, isna) in enumerate(mask_nan.items()):
            if isna and run_start is None:
                run_start = ts
            if run_start is not None and (not isna or i == len(mask_nan)-1):
                run_end = merged.index[i-1] if not isna else ts
                cnt = col.loc[run_start:run_end].isna().sum()
                ranges.append((run_start, run_end, int(cnt)))
                run_start = None

    feature_null_ranges[c] = ranges
    feature_nan_counts[c] = n_nan
    feature_range_counts[c] = len(ranges)

# Resumen ordenado por cantidad de NaNs
if feature_nan_counts:
    summary = (
        pd.DataFrame({
            "nan_count": pd.Series(feature_nan_counts, dtype="int"),
            "n_ranges": pd.Series(feature_range_counts, dtype="int"),
        })
        .sort_values(["nan_count","n_ranges"], ascending=[False, True])
    )
else:
    summary = pd.DataFrame(columns=["nan_count","n_ranges"])

print("Rangos de NaNs por feature (resumen)")
display(summary.head(50))

# Mostrar rangos completos para las top-K features con más NaNs
TOP_K_SHOW = 20
print(f"\nDetalle de rangos (top {TOP_K_SHOW} features por NaNs)")
for feat in summary.head(TOP_K_SHOW).index:
    print(f"\nFeature: {feat}")
    rows = feature_null_ranges.get(feat, [])
    print(f"  Total NaNs: {feature_nan_counts.get(feat, 0)} | Rangos: {len(rows)}")
    # Mostrar hasta 40 rangos con conteo
    MAX_RANGES_PRINT = 40
    for i, (s, e, cnt) in enumerate(rows[:MAX_RANGES_PRINT], start=1):
        print(f"   {i:>3}. {s} → {e}   (n={cnt})")
    if len(rows) > MAX_RANGES_PRINT:
        print(f"   ... ({len(rows) - MAX_RANGES_PRINT} rangos más)")

Rangos de NaNs por feature (resumen)


,nan_count,n_ranges
emae_desestacionalizado_spec_centroid_diff3,45,15
emae_original_spec_centroid_diff3,43,18
emae_tendencia_ciclo_spec_centroid_diff3,42,17
inflacion_spec_centroid_diff3,42,17
emae_original_spec_centroid_diff1,37,15
emae_desestacionalizado_spec_centroid_diff1,36,12
emae_tendencia_ciclo_spec_centroid_diff1,36,15
inflacion_spec_centroid_diff1,35,14
emae_desestacionalizado_spec_centroid,31,10
emae_tendencia_ciclo_spec_centroid,30,11



Detalle de rangos (top 20 features por NaNs)

Feature: emae_desestacionalizado_spec_centroid_diff3
  Total NaNs: 45 | Rangos: 15
     1. 2022-12-01 00:00:00 → 2022-12-02 00:00:00   (n=2)
     2. 2022-12-05 00:00:00 → 2022-12-07 00:00:00   (n=3)
     3. 2022-12-12 00:00:00 → 2022-12-16 00:00:00   (n=5)
     4. 2022-12-19 00:00:00 → 2022-12-23 00:00:00   (n=5)
     5. 2022-12-26 00:00:00 → 2022-12-30 00:00:00   (n=5)
     6. 2023-01-02 00:00:00 → 2023-01-03 00:00:00   (n=2)
     7. 2023-09-28 00:00:00 → 2023-09-29 00:00:00   (n=2)
     8. 2023-10-03 00:00:00 → 2023-10-04 00:00:00   (n=2)
     9. 2024-05-29 00:00:00 → 2024-05-31 00:00:00   (n=3)
    10. 2024-06-03 00:00:00 → 2024-06-05 00:00:00   (n=3)
    11. 2024-07-29 00:00:00 → 2024-08-02 00:00:00   (n=5)
    12. 2024-08-05 00:00:00 → 2024-08-05 00:00:00   (n=1)
    13. 2025-07-29 00:00:00 → 2025-08-01 00:00:00   (n=4)
    14. 2025-08-04 00:00:00 → 2025-08-05 00:00:00   (n=2)
    15. 2025-08-29 00:00:00 → 2025-08-29 00:00:00   (n=1)


Los rangos en realidad son partidos por algunos fines de semana...

Se hace un corte del primer mes del periodo, según lo previsto, para evitar nulls en el comienzo por el FE.

In [28]:
df_by_date.join(FE_ALL.set_index(DATE_COL), how="left").isnull().sum().sum()

2912

In [30]:
df_by_date.join(FE_ALL.set_index(DATE_COL), how="left").iloc[20:].isnull().sum().sum()

258

Se reducen los nulls en un 90 %

In [32]:
df_out = df_by_date.join(FE_ALL.set_index(DATE_COL), how="left").iloc[20:]

# Cálculo de rangos por feature
feature_null_ranges = {}
feature_nan_counts = {}
feature_range_counts = {}

for c in fe_cols:
    col = df_out[c]
    mask_nan = col.isna()
    n_nan = int(mask_nan.sum())
    if n_nan == 0:
        continue
    if '_ranges_from_mask' in globals():
        ranges = _ranges_from_mask(df_out.index, mask_nan, step=step)
    else:
        # fallback simple si no está _ranges_from_mask
        ranges = []
        run_start = None
        for i, (ts, isna) in enumerate(mask_nan.items()):
            if isna and run_start is None:
                run_start = ts
            if run_start is not None and (not isna or i == len(mask_nan)-1):
                run_end = df_out.index[i-1] if not isna else ts
                cnt = col.loc[run_start:run_end].isna().sum()
                ranges.append((run_start, run_end, int(cnt)))
                run_start = None

    feature_null_ranges[c] = ranges
    feature_nan_counts[c] = n_nan
    feature_range_counts[c] = len(ranges)

# Resumen ordenado por cantidad de NaNs
if feature_nan_counts:
    summary = (
        pd.DataFrame({
            "nan_count": pd.Series(feature_nan_counts, dtype="int"),
            "n_ranges": pd.Series(feature_range_counts, dtype="int"),
        })
        .sort_values(["nan_count","n_ranges"], ascending=[False, True])
    )
else:
    summary = pd.DataFrame(columns=["nan_count","n_ranges"])

print("Rangos de NaNs por feature (resumen)")
display(summary.head(50))

# Mostrar rangos completos para las top-K features con más NaNs
TOP_K_SHOW = 20
print(f"\nDetalle de rangos (top {TOP_K_SHOW} features por NaNs)")
for feat in summary.head(TOP_K_SHOW).index:
    print(f"\nFeature: {feat}")
    rows = feature_null_ranges.get(feat, [])
    print(f"  Total NaNs: {feature_nan_counts.get(feat, 0)} | Rangos: {len(rows)}")
    # Mostrar hasta 40 rangos con conteo
    MAX_RANGES_PRINT = 40
    for i, (s, e, cnt) in enumerate(rows[:MAX_RANGES_PRINT], start=1):
        print(f"   {i:>3}. {s} → {e}   (n={cnt})")
    if len(rows) > MAX_RANGES_PRINT:
        print(f"   ... ({len(rows) - MAX_RANGES_PRINT} rangos más)")

Rangos de NaNs por feature (resumen)


,nan_count,n_ranges
emae_desestacionalizado_spec_centroid_diff3,25,10
emae_original_spec_centroid_diff3,23,13
emae_tendencia_ciclo_spec_centroid_diff3,22,12
inflacion_spec_centroid_diff3,22,12
emae_original_spec_centroid_diff1,17,10
emae_desestacionalizado_spec_centroid_diff1,16,7
emae_tendencia_ciclo_spec_centroid_diff1,16,10
inflacion_spec_centroid_diff1,15,9
emae_desestacionalizado_spec_centroid,12,5
emae_tendencia_ciclo_spec_centroid,10,6



Detalle de rangos (top 20 features por NaNs)

Feature: emae_desestacionalizado_spec_centroid_diff3
  Total NaNs: 25 | Rangos: 10
     1. 2023-01-02 00:00:00 → 2023-01-03 00:00:00   (n=2)
     2. 2023-09-28 00:00:00 → 2023-09-29 00:00:00   (n=2)
     3. 2023-10-03 00:00:00 → 2023-10-04 00:00:00   (n=2)
     4. 2024-05-29 00:00:00 → 2024-05-31 00:00:00   (n=3)
     5. 2024-06-03 00:00:00 → 2024-06-05 00:00:00   (n=3)
     6. 2024-07-29 00:00:00 → 2024-08-02 00:00:00   (n=5)
     7. 2024-08-05 00:00:00 → 2024-08-05 00:00:00   (n=1)
     8. 2025-07-29 00:00:00 → 2025-08-01 00:00:00   (n=4)
     9. 2025-08-04 00:00:00 → 2025-08-05 00:00:00   (n=2)
    10. 2025-08-29 00:00:00 → 2025-08-29 00:00:00   (n=1)

Feature: emae_original_spec_centroid_diff3
  Total NaNs: 23 | Rangos: 13
     1. 2023-01-02 00:00:00 → 2023-01-04 00:00:00   (n=3)
     2. 2023-01-27 00:00:00 → 2023-01-27 00:00:00   (n=1)
     3. 2023-01-30 00:00:00 → 2023-02-03 00:00:00   (n=5)
     4. 2023-05-31 00:00:00 → 2023-05-31 0

Imputacion de los nulls pendientes

Forward Fill o fill 0.0

In [33]:
df_out.isnull().sum().sum()

258

In [34]:
df_out.ffill().fillna(0.0).isnull().sum().sum()

0

In [35]:
df_out = df_out.ffill().fillna(0.0)

#### Saving FE dataset <a id="saving-fe"></a>
Persistencia del dataset de *feature engineering* (CSV/Parquet)

In [36]:
if df_out is None:
    print("[INFO] Saltando: no hay df.")
else:
    df_out.to_csv(OUTPUT_PATH, index=True)
    print(f"Guardado CSV: {OUTPUT_PATH}")

Guardado CSV: ../inputs/dataset_v2_with_fe.csv
